In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/ecommerce_customer_churn_dataset.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4


In [2]:
df_fe = df.copy()

print("Original shape:", df.shape)
print("Feature engineering dataframe shape:", df_fe.shape)

Original shape: (50000, 25)
Feature engineering dataframe shape: (50000, 25)


## Purchase Frequency

In [3]:
df_fe["Purchase_Frequency"] = (
    df_fe["Total_Purchases"] /
    (df_fe["Membership_Years"])
)

In [4]:
df_fe[[
    "Total_Purchases",
    "Membership_Years",
    "Purchase_Frequency"
]].head()

,Total_Purchases,Membership_Years,Purchase_Frequency
0,9.0,2.9,3.103448
1,19.5,1.6,12.187500
2,9.1,2.9,3.137931
3,15.0,2.6,5.769231
4,32.5,3.1,10.483871


In [5]:
df_fe["Purchase_Frequency"].describe()

count    50000.000000
mean         9.081994
std         21.289030
min        -16.250000
25%          2.608696
50%          4.750000
75%          9.090909
max       1162.750736
Name: Purchase_Frequency, dtype: float64

### Recently Active Customer

In [6]:
df_fe["Is_Recently_Active"] = (
    df_fe["Days_Since_Last_Purchase"] <= 30
).astype(int)

In [7]:
df_fe["Is_Recently_Active"].value_counts()

Is_Recently_Active
1    30108
0    19892
Name: count, dtype: int64

In [8]:
pd.crosstab(
    df_fe["Is_Recently_Active"],
    df_fe["Churned"],
    normalize="index"
)

Churned,0,1
Is_Recently_Active,,
0,0.650613,0.349387
1,0.750897,0.249103


### High Cart Abandonment

In [9]:
df_fe["High_Cart_Abandonment"] = (
    df_fe["Cart_Abandonment_Rate"] >= 60
).astype(int)

In [10]:
df_fe["High_Cart_Abandonment"].value_counts()

High_Cart_Abandonment
0    27245
1    22755
Name: count, dtype: int64

In [11]:
pd.crosstab(
    df_fe["High_Cart_Abandonment"],
    df_fe["Churned"],
    normalize="index"
)

Churned,0,1
High_Cart_Abandonment,,
0,0.806423,0.193577
1,0.596748,0.403252


#### Check new features

In [13]:
new_features = [
    "Purchase_Frequency",
    "Is_Recently_Active",
    "High_Cart_Abandonment"
]

df_fe[new_features].describe()

,Purchase_Frequency,Is_Recently_Active,High_Cart_Abandonment
count,50000.000000,50000.000000,50000.000000
mean,9.081994,0.602160,0.455100
std,21.289030,0.489457,0.497985
min,-16.250000,0.000000,0.000000
25%,2.608696,0.000000,0.000000
50%,4.750000,1.000000,0.000000
75%,9.090909,1.000000,1.000000
max,1162.750736,1.000000,1.000000


In [ ]:
feature_correlations = (
    df_fe[new_features + ["Churned"]]
    .corr()["Churned"]
    .drop("Churned")
    .sort_values(key=abs, ascending=False)
)

print(feature_correlations)

In [ ]:
print("Missing values in new features:")
print(df_fe[new_features].isnull().sum())

## Feature Engineering Summary

Four behavioral features were created:

1. **Purchase_Frequency**
   - Measures purchase activity relative to membership duration.
   - Formula: Total_Purchases / (Membership_Years + 1)

2. **Is_Recently_Active**
   - Indicates whether the customer made a purchase within the last 30 days.
   - 1 = recently active
   - 0 = not recently active


3. **High_Cart_Abandonment**
   - Indicates customers with a cart abandonment rate of at least 60%.
   - 1 = high cart abandonment
   - 0 = otherwise

These features were created based on customer purchasing behavior and observations from the exploratory data analysis. Their usefulness will be evaluated during model development rather than assuming that feature engineering automatically improves model performance.